# Auditeur de Coherence Medicale
**Verification de la coherence entre une radiographie X-Ray et son compte-rendu radiologique.**

Architecture multimodale :
- Encodeur vision : DenseNet-121
- Encodeur texte : BioClinicalBERT
- Tete de classification : coherent / incoherent

Dataset : CheXpert-v1.0-small (images) + CheXpert Plus CSV (rapports) - lus depuis Google Drive

---
> Verifier que le runtime est regle sur **GPU T4** : Execution > Modifier le type d'execution > GPU

## Cellule 1 - Verification du GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print("GPU detecte :", gpu_name)
    print("VRAM totale :", round(vram_total, 1), "Go")
else:
    print("Aucun GPU detecte - verifier le runtime Colab (Execution > Modifier le type d'execution > GPU)")

## Cellule 2 - Installation des dependances

In [ ]:
!pip install -q transformers==4.41.0 torchvision Pillow pandas scikit-learn tqdm

## Cellule 3 - Montage de Google Drive

Une fenetre va s'ouvrir pour autoriser Colab a acceder a ton Drive.
Clique sur **Autoriser** et connecte-toi avec ton compte Google.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Verification que les fichiers sont bien presents
CHEXPERT_DIR     = '/content/drive/MyDrive/CheXpert-v1.0-small'
CHEXPERT_PLUS_CSV = '/content/drive/MyDrive/df_chexpert_plus_240401.csv'

# Verification du dossier images
if os.path.exists(CHEXPERT_DIR):
    print("Dossier CheXpert trouve :", CHEXPERT_DIR)
    print("Contenu :", os.listdir(CHEXPERT_DIR))
else:
    print("ERREUR : dossier CheXpert introuvable.")
    print("Verifier que le dossier s'appelle bien CheXpert-v1.0-small dans Mon Drive.")
    print("Contenu de Mon Drive :", os.listdir('/content/drive/MyDrive/'))

# Verification du CSV
if os.path.exists(CHEXPERT_PLUS_CSV):
    print("CSV CheXpert Plus trouve :", CHEXPERT_PLUS_CSV)
else:
    print("ERREUR : CSV CheXpert Plus introuvable.")
    print("Verifier que le fichier s'appelle bien df_chexpert_plus_240401.csv dans Mon Drive.")

## Cellule 4 - Imports et configuration

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import torchvision.transforms as transforms
import torchvision.models as models

from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Reproductibilite
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device utilise :", DEVICE)

# Hyperparametres - calibres pour T4 16 Go avec FP16
BATCH_SIZE    = 16
ACCUMULATION  = 4
MAX_TEXT_LEN  = 256
IMAGE_SIZE    = 224
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 10
NUM_WORKERS   = 2
NEG_RATIO     = 1.0

# Chemins Google Drive - definis en cellule 3
# CHEXPERT_DIR et CHEXPERT_PLUS_CSV sont deja definis

print("Configuration chargee.")

## Cellule 5 - Chargement et fusion des metadonnees

In [ ]:
# Chargement des CSV CheXpert
df_train = pd.read_csv(os.path.join(CHEXPERT_DIR, 'train.csv'))
df_valid = pd.read_csv(os.path.join(CHEXPERT_DIR, 'valid.csv'))
df_meta  = pd.concat([df_train, df_valid], ignore_index=True)

print("Metadonnees CheXpert :", len(df_meta), "entrees")
print("Colonnes :", df_meta.columns.tolist())
df_meta.head(3)

In [ ]:
# Chargement du CSV CheXpert Plus
df_plus = pd.read_csv(CHEXPERT_PLUS_CSV)
print("CheXpert Plus :", len(df_plus), "entrees")
print("Colonnes :", df_plus.columns.tolist())
df_plus.head(3)

In [ ]:
# Inspection des colonnes pour trouver la cle de fusion
# Cette cellule t'aide a identifier les bonnes colonnes avant de fusionner

print("=== Colonnes CheXpert ===")
print(df_meta.columns.tolist())
print("\nExemple Path :", df_meta['Path'].iloc[0])

print("\n=== Colonnes CheXpert Plus ===")
print(df_plus.columns.tolist())
print("\nPremiere ligne :")
print(df_plus.iloc[0])

In [ ]:
# Fusion des deux datasets
# Extraction de l'identifiant depuis le chemin
df_meta['study_id'] = df_meta['Path'].str.extract(r'(patient\d+/study\d+)')

# Adapter 'study_id' si le nom de colonne differe dans df_plus
# (verifier avec la cellule precedente)
df = df_meta.merge(df_plus, on='study_id', how='inner')
print("Apres fusion :", len(df), "paires image-rapport")

# Garder uniquement les vues frontales
if 'Frontal/Lateral' in df.columns:
    df = df[df['Frontal/Lateral'] == 'Frontal'].reset_index(drop=True)
    print("Apres filtrage vues frontales :", len(df), "paires")

# Colonne rapport - adapter si necessaire
REPORT_COL = 'report'
df = df.dropna(subset=[REPORT_COL]).reset_index(drop=True)
print("Apres suppression rapports vides :", len(df), "paires")

## Cellule 6 - Generation des paires coherentes / incoherentes

In [ ]:
def build_pairs(df, neg_ratio=1.0, seed=42):
    rng = np.random.default_rng(seed)
    pairs = []

    # Paires positives
    for _, row in df.iterrows():
        pairs.append({
            'image_path': row['Path'],
            'report':     row[REPORT_COL],
            'label':      1
        })

    # Paires negatives - rapport d'un autre patient
    n_neg = int(len(df) * neg_ratio)
    indices = np.arange(len(df))
    for i in rng.choice(len(df), size=n_neg, replace=False):
        other_indices = indices[indices != i]
        j = rng.choice(other_indices)
        pairs.append({
            'image_path': df.iloc[i]['Path'],
            'report':     df.iloc[j][REPORT_COL],
            'label':      0
        })

    df_pairs = pd.DataFrame(pairs).sample(frac=1, random_state=seed).reset_index(drop=True)
    print("Total paires :", len(df_pairs),
          "(", df_pairs['label'].sum(), "coherentes,",
          (df_pairs['label']==0).sum(), "incoherentes)")
    return df_pairs


df_pairs = build_pairs(df, neg_ratio=NEG_RATIO)

df_trainval, df_test = train_test_split(df_pairs, test_size=0.10,
                                         stratify=df_pairs['label'], random_state=SEED)
df_train, df_val     = train_test_split(df_trainval, test_size=0.11,
                                         stratify=df_trainval['label'], random_state=SEED)

print("Train :", len(df_train), "| Val :", len(df_val), "| Test :", len(df_test))

## Cellule 7 - Dataset PyTorch

In [ ]:
BERT_MODEL_NAME = 'emilyalsentzer/Bio_ClinicalBERT'
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


class CheXpertCoherenceDataset(Dataset):
    def __init__(self, df, img_root, transform, tokenizer, max_len):
        self.df        = df.reset_index(drop=True)
        self.img_root  = img_root
        self.transform = transform
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_root, row['image_path'])
        image = Image.open(img_path).convert('RGB')
        image = self.transform(image)

        encoding = self.tokenizer(
            str(row['report']),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'image':          image,
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label':          torch.tensor(row['label'], dtype=torch.long)
        }


train_dataset = CheXpertCoherenceDataset(df_train, CHEXPERT_DIR, train_transform, tokenizer, MAX_TEXT_LEN)
val_dataset   = CheXpertCoherenceDataset(df_val,   CHEXPERT_DIR, val_transform,   tokenizer, MAX_TEXT_LEN)
test_dataset  = CheXpertCoherenceDataset(df_test,  CHEXPERT_DIR, val_transform,   tokenizer, MAX_TEXT_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print("Batches - Train :", len(train_loader), "| Val :", len(val_loader), "| Test :", len(test_loader))

## Cellule 8 - Architecture du modele multimodal

In [ ]:
class MedicalCoherenceAuditor(nn.Module):
    def __init__(self, bert_model_name, freeze_bert_layers=6, dropout=0.3):
        super().__init__()

        # Encodeur vision : DenseNet-121
        densenet = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        self.vision_encoder = densenet.features
        self.vision_pool    = nn.AdaptiveAvgPool2d((1, 1))
        vision_dim          = 1024

        for name, param in self.vision_encoder.named_parameters():
            if 'denseblock1' in name or 'denseblock2' in name:
                param.requires_grad = False

        # Encodeur texte : BioClinicalBERT
        self.text_encoder = AutoModel.from_pretrained(bert_model_name)
        text_dim          = self.text_encoder.config.hidden_size

        for layer in self.text_encoder.encoder.layer[:freeze_bert_layers]:
            for param in layer.parameters():
                param.requires_grad = False

        # Couches de projection
        proj_dim = 512
        self.vision_proj = nn.Sequential(
            nn.Linear(vision_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.ReLU()
        )
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.ReLU()
        )

        # Tete de classification
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(proj_dim * 2, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 2)
        )

    def forward(self, image, input_ids, attention_mask):
        v = self.vision_encoder(image)
        v = self.vision_pool(v).flatten(1)
        v = self.vision_proj(v)

        t = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        t = t.last_hidden_state[:, 0, :]
        t = self.text_proj(t)

        fused  = torch.cat([v, t], dim=1)
        logits = self.classifier(fused)
        return logits


model = MedicalCoherenceAuditor(
    bert_model_name=BERT_MODEL_NAME,
    freeze_bert_layers=6,
    dropout=0.3
).to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Parametres totaux    :", f"{total:,}")
print("Parametres entraines :", f"{trainable:,}", f"({100*trainable/total:.1f}%)")

## Cellule 9 - Entrainement avec mixed precision (FP16)

In [ ]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
criterion = nn.CrossEntropyLoss()
scaler    = GradScaler()


def train_epoch(model, loader, optimizer, criterion, scaler, accumulation_steps):
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(loader, desc='Train', leave=False)):
        image     = batch['image'].to(DEVICE)
        input_ids = batch['input_ids'].to(DEVICE)
        attn_mask = batch['attention_mask'].to(DEVICE)
        labels    = batch['label'].to(DEVICE)

        with autocast():
            logits = model(image, input_ids, attn_mask)
            loss   = criterion(logits, labels) / accumulation_steps

        scaler.scale(loss).backward()

        if (step + 1) % accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * accumulation_steps
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader), acc


@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    for batch in tqdm(loader, desc='Val', leave=False):
        image     = batch['image'].to(DEVICE)
        input_ids = batch['input_ids'].to(DEVICE)
        attn_mask = batch['attention_mask'].to(DEVICE)
        labels    = batch['label'].to(DEVICE)

        with autocast():
            logits = model(image, input_ids, attn_mask)
            loss   = criterion(logits, labels)

        total_loss += loss.item()
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader), acc


# Dossier de sauvegarde sur Drive
SAVE_DIR = '/content/drive/MyDrive/auditeur_coherence_medicale'
os.makedirs(SAVE_DIR, exist_ok=True)
best_model_path = os.path.join(SAVE_DIR, 'best_model.pt')

best_val_acc = 0.0
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, scaler, ACCUMULATION)
    val_loss,   val_acc   = eval_epoch(model, val_loader, criterion)
    scheduler.step()

    history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                    'val_loss': val_loss, 'val_acc': val_acc})

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS}  "
          f"Train loss={train_loss:.4f} acc={train_acc:.4f}  "
          f"Val loss={val_loss:.4f} acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print("  -> Meilleur modele sauvegarde sur Drive (val_acc={:.4f})".format(best_val_acc))

print("\nEntrainement termine. Meilleure precision validation :", round(best_val_acc, 4))

## Cellule 10 - Evaluation sur le jeu de test

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Test'):
        image     = batch['image'].to(DEVICE)
        input_ids = batch['input_ids'].to(DEVICE)
        attn_mask = batch['attention_mask'].to(DEVICE)
        labels    = batch['label'].to(DEVICE)

        with autocast():
            logits = model(image, input_ids, attn_mask)

        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("\nRapport de classification sur le jeu de test :")
print(classification_report(all_labels, all_preds,
                             target_names=['Incoherent', 'Coherent'],
                             digits=4))

## Cellule 11 - Courbes d'apprentissage

In [ ]:
import matplotlib.pyplot as plt

df_hist = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(df_hist['epoch'], df_hist['train_loss'], label='Train', marker='o', markersize=4)
axes[0].plot(df_hist['epoch'], df_hist['val_loss'],   label='Val',   marker='s', markersize=4)
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(df_hist['epoch'], df_hist['train_acc'], label='Train', marker='o', markersize=4)
axes[1].plot(df_hist['epoch'], df_hist['val_acc'],   label='Val',   marker='s', markersize=4)
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'courbes_apprentissage.png'), dpi=120)
plt.show()
print("Courbes sauvegardees sur Drive.")

## Cellule 12 - Inference sur une paire quelconque

In [ ]:
def predict_coherence(image_path, report_text, model, tokenizer, transform, device):
    model.eval()

    image = Image.open(image_path).convert('RGB')
    image = transform(image).unsqueeze(0).to(device)

    encoding = tokenizer(
        report_text,
        max_length=MAX_TEXT_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids = encoding['input_ids'].to(device)
    attn_mask = encoding['attention_mask'].to(device)

    with torch.no_grad(), autocast():
        logits = model(image, input_ids, attn_mask)
        probs  = F.softmax(logits, dim=1).cpu().numpy()[0]

    label      = 'Coherent' if probs[1] >= 0.5 else 'Incoherent'
    confidence = max(probs)
    return label, confidence, {'incoherent': round(float(probs[0]), 4), 'coherent': round(float(probs[1]), 4)}


# Test sur un exemple du jeu de test
sample = df_test.sample(1).iloc[0]
img_path = os.path.join(CHEXPERT_DIR, sample['image_path'])
rapport  = sample['report']

label, confidence, scores = predict_coherence(
    img_path, rapport, model, tokenizer, val_transform, DEVICE
)

print("Verdict    :", label)
print("Confiance  :", f"{confidence:.2%}")
print("Scores     :", scores)
print("Label reel :", 'Coherent' if sample['label'] == 1 else 'Incoherent')

## Cellule 13 - Sauvegarde de l'historique sur Drive

In [ ]:
pd.DataFrame(history).to_csv(os.path.join(SAVE_DIR, 'training_history.csv'), index=False)
print("Tout est sauvegarde dans :", SAVE_DIR)
print("Contenu :", os.listdir(SAVE_DIR))